# Maize Yield Prediction in Central Europe

This notebook implements the methodology from the MDPI paper:
**"Data Mining and Machine Learning Algorithms for Optimizing Maize Yield Forecasting in Central Europe"**.

It predicts maize yield using various agricultural and climate data scenarios and compares the performance of 4 machine learning algorithms (Bagging, Decision Table proxy, Random Forest, and ANN-MLP). Crucially, to replicate WEKA's implicit data handling, both the feature vectors and the target yield vectors are Standardized, and the `lbfgs` optimizer is applied to ensure full convergence accuracy.

In [14]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Models
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor

## 1. Load Dataset


In [16]:
# Load dataset
df = pd.read_csv('paper_final_maize_climate_dataset.csv')
display(df.head())

# Define Target Output (Y)
y = df['Maize_avg_yield']

# Define the full feature set
features = df[['Maize_sown_area', 'Maize_production', 'Tmean', 'PRCP', 'RD', 'FD', 'HD']]
features.columns = ['AREA', 'PROD', 'Tmean', 'PRCP', 'RD', 'FD', 'HD']

,year,Tmean,PRCP,RD,FD,HD,Heat_Stress,Sunshine,Rain_Intensity,Dry_Index,Temp_Index,Maize_sown_area,Maize_production,Maize_avg_yield,Yield_computed
0,1921,10.2,466,135,107,25,9,0.0,3.451852,0.0,255.0,876974,805295,920,0.918266
1,1922,8.9,674,161,118,19,2,0.0,4.186335,0.0,169.1,989459,1237674,1250,1.250859
2,1923,10.4,605,162,76,14,1,0.0,3.734568,0.0,145.6,995177,1250942,1260,1.257005
3,1924,9.1,577,148,128,9,1,0.0,3.898649,0.0,81.9,1003343,1882807,1880,1.876534
4,1925,9.8,717,163,104,5,0,0.0,4.398773,0.0,49.0,1082099,2234548,2070,2.065013


## 2. Scenarios
- **SC1**: AREA+PROD+Tmean+PRCP+RD+FD+HD
- **SC2**: AREA+PROD
- **SC3**: Tmean+PRCP+RD+FD+HD
- **SC4**: AREA+PROD+Tmean+PRCP

In [17]:
scenarios = {
    'SC1': ['AREA', 'PROD', 'Tmean', 'PRCP', 'RD', 'FD', 'HD'],
    'SC2': ['AREA', 'PROD'],
    'SC3': ['Tmean', 'PRCP', 'RD', 'FD', 'HD'],
    'SC4': ['AREA', 'PROD', 'Tmean', 'PRCP']
}

## 3. Evaluation Metrics

In [18]:
def calculate_metrics(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    r = np.corrcoef(y_true, y_pred)[0, 1]
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    y_mean = np.mean(y_true)
    rae = (np.sum(np.abs(y_pred - y_true)) / np.sum(np.abs(y_mean - y_true))) * 100
    rrse = np.sqrt(np.sum((y_pred - y_true)**2) / np.sum((y_mean - y_true)**2)) * 100
    
    return {'r': r, 'MAE': mae, 'RMSE': rmse, 'RAE': rae, 'RRSE': rrse}

## 4. Models


In [19]:
def get_models():
    return {
        'Bagging (BG)': BaggingRegressor(estimator=DecisionTreeRegressor(), n_estimators=100, random_state=1),
        'Decision Table (DT Proxy)': DecisionTreeRegressor(max_depth=3, random_state=1),
        'Random Forest (RF)': RandomForestRegressor(n_estimators=100, max_depth=None, random_state=1),
        'ANN-MLP': MLPRegressor(hidden_layer_sizes=(50,), activation='logistic', 
                                solver='lbfgs', max_iter=2000, random_state=1)
    }

## 5. Experiment 1: 80-20 Split


In [20]:
results_split = []

for sc_name, sc_features in scenarios.items():
    X = features[sc_features]
    
    # Scale Features & Target
    scaler_x = StandardScaler()
    scaler_y = StandardScaler()
    
    X_scaled = scaler_x.fit_transform(X)
    y_scaled = scaler_y.fit_transform(y.values.reshape(-1, 1)).ravel()
    
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_scaled, test_size=0.20, random_state=1)
    
    models = get_models()
    for model_name, model in models.items():
        model.fit(X_train, y_train)
        y_pred_scaled = model.predict(X_test)
        
        # Inverse scale back to true yield domains for true error calculation
        y_test_inv = scaler_y.inverse_transform(y_test.reshape(-1, 1)).ravel()
        y_pred_inv = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
        
        metrics = calculate_metrics(y_test_inv, y_pred_inv)
        metrics['Scenario'] = sc_name
        metrics['Model'] = model_name
        metrics['Phase'] = 'Testing (80-20)'
        results_split.append(metrics)

df_results_split = pd.DataFrame(results_split)
df_results_split = df_results_split[['Phase', 'Scenario', 'Model', 'r', 'MAE', 'RMSE', 'RAE', 'RRSE']]
display(df_results_split)

,Phase,Scenario,Model,r,MAE,RMSE,RAE,RRSE
0,Testing (80-20),SC1,Bagging (BG),0.977952,246.035000,419.808202,15.158806,21.904516
1,Testing (80-20),SC1,Decision Table (DT Proxy),0.963932,323.490351,512.445712,19.931016,26.738104
2,Testing (80-20),SC1,Random Forest (RF),0.976890,248.620000,430.336840,15.318074,22.453874
3,Testing (80-20),SC1,ANN-MLP,0.994538,133.339773,210.785441,8.215383,10.998244
4,Testing (80-20),SC2,Bagging (BG),0.982439,224.145000,396.458514,13.810111,20.686190
5,Testing (80-20),SC2,Decision Table (DT Proxy),0.963932,323.490351,512.445712,19.931016,26.738104
6,Testing (80-20),SC2,Random Forest (RF),0.982778,217.315000,390.894476,13.389298,20.395872
7,Testing (80-20),SC2,ANN-MLP,0.998199,76.508429,119.978040,4.713868,6.260147
8,Testing (80-20),SC3,Bagging (BG),0.316317,1656.320000,1902.349765,102.049844,99.259737
9,Testing (80-20),SC3,Decision Table (DT Proxy),0.094352,1835.856154,2288.813673,113.111497,119.424434


## 6. Experiment 2: 10-Fold CV


In [21]:
results_cv = []
kf = KFold(n_splits=10, shuffle=True, random_state=1)

for sc_name, sc_features in scenarios.items():
    X = features[sc_features]
    scaler_x = StandardScaler()
    scaler_y = StandardScaler()
    
    X_scaled = scaler_x.fit_transform(X)
    y_scaled = scaler_y.fit_transform(y.values.reshape(-1, 1)).ravel()
    
    models = get_models()
    for model_name, model in models.items():
        cv_preds = np.zeros_like(y_scaled)
        
        for train_idx, test_idx in kf.split(X_scaled):
            X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
            y_train = y_scaled[train_idx]
            model.fit(X_train, y_train)
            cv_preds[test_idx] = model.predict(X_test)
            
        # Inverse scale
        y_true_inv = scaler_y.inverse_transform(y_scaled.reshape(-1, 1)).ravel()
        cv_preds_inv = scaler_y.inverse_transform(cv_preds.reshape(-1, 1)).ravel()
        
        metrics = calculate_metrics(y_true_inv, cv_preds_inv)
        metrics['Scenario'] = sc_name
        metrics['Model'] = model_name
        metrics['Phase'] = '10-Fold CV'
        results_cv.append(metrics)

df_results_cv = pd.DataFrame(results_cv)
df_results_cv = df_results_cv[['Phase', 'Scenario', 'Model', 'r', 'MAE', 'RMSE', 'RAE', 'RRSE']]
display(df_results_cv)

,Phase,Scenario,Model,r,MAE,RMSE,RAE,RRSE
0,10-Fold CV,SC1,Bagging (BG),0.984284,257.903061,372.026938,14.186259,17.835487
1,10-Fold CV,SC1,Decision Table (DT Proxy),0.962020,392.831728,570.440441,21.608168,27.347705
2,10-Fold CV,SC1,Random Forest (RF),0.984533,257.861224,369.474230,14.183958,17.713106
3,10-Fold CV,SC1,ANN-MLP,0.995654,129.666200,194.972761,7.132441,9.347264
4,10-Fold CV,SC2,Bagging (BG),0.989314,207.295918,309.805864,11.402554,14.852522
5,10-Fold CV,SC2,Decision Table (DT Proxy),0.966776,373.781275,538.233944,20.560276,25.803681
6,10-Fold CV,SC2,Random Forest (RF),0.989781,202.031633,304.600773,11.112986,14.602983
7,10-Fold CV,SC2,ANN-MLP,0.998423,70.645780,117.170271,3.885954,5.617305
8,10-Fold CV,SC3,Bagging (BG),0.352924,1679.429592,1964.236397,92.378987,94.168214
9,10-Fold CV,SC3,Decision Table (DT Proxy),0.269290,1701.667212,2102.729116,93.602193,100.807746


# Barley Yield Prediction in Central Europe

In [ ]:
# Load dataset
df = pd.read_csv('paper_final_barley_climate_dataset.csv')
display(df.head())

# Define Target Output (Y)
y = df['Barley_avg_yield']

# Define the full feature set
features = df[['Barley_sown_area', 'Barley_production', 'Tmean', 'PRCP', 'RD', 'FD', 'HD']]
features.columns = ['AREA', 'PROD', 'Tmean', 'PRCP', 'RD', 'FD', 'HD']